In [ ]:
# SVR tuning, multiple targets, and pooled denormalized metrics (NoIntp+ManVa)

import os
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics as sm
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR


# Configuration
names = ["BMP", "COD", "NR", "PR"]

inpt_nums = {
    "BMP": 10,
    "COD": 10,
    "NR": 10,
    "PR": 10,
}

niter = 200
initp = 20

k_out = 5
fo_start = 1

parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv2/"


def make_regressor(log_c, log_epsilon, log_gamma):
    # hyperparameters are searched on a log10 scale
    return SVR(
        kernel="rbf",
        C=10 ** log_c,
        epsilon=10 ** log_epsilon,
        gamma=10 ** log_gamma,
    )


def relative_error_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    error = actual - np.asarray(predicted, dtype=float)

    return np.divide(
        error * 100.0,
        actual,
        out=np.full_like(error, np.nan),
        where=actual != 0,
    )


def metric_values(actual, predicted):
    mse = sm.mean_squared_error(actual, predicted)

    return {
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(mse, 3),
        "RMSE": round(float(np.sqrt(mse)), 3),
        "R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
    }


def pooled_metrics_row(group, scope, fold_out="All"):
    actual = group["Actual(real)"].to_numpy(dtype=float)
    predicted = group["Prediction(real)"].to_numpy(dtype=float)
    residual = actual - predicted
    relative_error = relative_error_percent(actual, predicted)

    return {
        "Name": group["Name"].iloc[0],
        "Scope": scope,
        "Fold_out": fold_out,
        "N_predictions": len(group),
        "Pooled_R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(sm.mean_squared_error(actual, predicted), 3),
        "RE_mean": round(float(np.mean(residual)), 3),
        "RE%_mean": round(float(np.nanmean(relative_error)), 3),
    }


for name in names:
    print(f"===== Running {name} =====")
    inpt_num = inpt_nums[name]

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path1 = os.path.join(
        parent_dir,
        f"SVR {name} Best Models_{timestamp}",
    )
    os.makedirs(path1, exist_ok=True)

    performance_rows = []
    pooled_rows = []

    for fold_no_out in range(fo_start, k_out + 1):
        # Outer held-out test data.
        data2 = pd.read_csv(os.path.join(
            parent_dir,
            f"Testing_data_ori_{name}-fold{fold_no_out}.csv",
        ))

        # Remaining original folds form the outer-training data.
        data = pd.read_csv(parent_dir + 'Training_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
        # extra_frames = []
        # for fold_num in range(1, k_out + 1):
        #     if fold_num == fold_no_out:
        #         continue
        #     extra_frames.append(pd.read_csv(os.path.join(
        #         parent_dir,
        #         f"Testing_data_ori_{name}-fold{fold_num}.csv",
        #     )))
        # data = pd.concat(extra_frames, axis=0, ignore_index=True)

        # Random 20% of outer-training data as validation.
        data, data3 = train_test_split(
            data,
            test_size=0.20,
            random_state=42,
            shuffle=True,
        )

        # Fit only on training data; transform all three datasets.
        scaler = StandardScaler()
        scaler.fit(data)

        dataset1 = scaler.transform(data)
        dataset2 = scaler.transform(data2)
        dataset3 = scaler.transform(data3)

        X_training = dataset1[:, :inpt_num]
        Y_training = dataset1[:, inpt_num]

        X_testing = dataset2[:, :inpt_num]
        Y_testing = dataset2[:, inpt_num]

        X_valid = dataset3[:, :inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # Bayesian optimization of C, epsilon, gamma (validation R2).
        def svr_objective(log_c, log_epsilon, log_gamma):
            model = make_regressor(log_c, log_epsilon, log_gamma)
            model.fit(X_training, Y_training)
            prediction = model.predict(X_valid)
            return sm.r2_score(Y_valid, prediction)

        optimizer = BayesianOptimization(
            f=svr_objective,
            pbounds={
                "log_c": (-2, 3),         # C: 0.01 - 1000
                "log_epsilon": (-3, 0),   # epsilon: 0.001 - 1 (normalized target)
                "log_gamma": (-4, 1),     # gamma: 1e-4 - 10
            },
            random_state=1,
        )
        optimizer.maximize(
            n_iter=niter,
            init_points=initp,
        )
        best_params = optimizer.max["params"]

        model = make_regressor(
            best_params["log_c"],
            best_params["log_epsilon"],
            best_params["log_gamma"],
        )
        model.fit(X_training, Y_training)
        prediction = model.predict(X_testing)

        model_name = f"SVR_model_{name}_kote_{fold_no_out}"
        joblib.dump(
            model,
            os.path.join(path1, model_name + ".joblib"),
        )

        # Convert target values and predictions back to original units.
        ymean = scaler.mean_[inpt_num]
        yscale = scaler.scale_[inpt_num]

        actual_real = Y_testing * yscale + ymean
        predicted_real = prediction * yscale + ymean
        residual_real = actual_real - predicted_real
        residual_percent_real = relative_error_percent(
            actual_real,
            predicted_real,
        )

        normalized_metrics = metric_values(Y_testing, prediction)
        real_metrics = metric_values(actual_real, predicted_real)

        performance_rows.append({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "C": round(10 ** best_params["log_c"], 5),
            "epsilon": round(10 ** best_params["log_epsilon"], 5),
            "gamma": round(10 ** best_params["log_gamma"], 5),
            "Val_R2": round(optimizer.max["target"], 3),
            **normalized_metrics,
            "MAE_real": real_metrics["MAE"],
            "MSE_real": real_metrics["MSE"],
            "RMSE_real": real_metrics["RMSE"],
            "R2_real": real_metrics["R2"],
            "RE_mean_real": round(float(np.mean(residual_real)), 3),
            "RE%_mean_real": round(float(np.nanmean(residual_percent_real)), 3),
        })

        predictions = pd.DataFrame({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "Test_row": np.arange(len(Y_testing)),
            "Actual": Y_testing,
            "Prediction": prediction,
            "Actual(real)": actual_real,
            "Prediction(real)": predicted_real,
            "RErr(real)": residual_real,
            "RErr%(real)": residual_percent_real,
        })
        pooled_rows.append(predictions)

        predictions.to_csv(
            os.path.join(path1, f"Predictions_{name}_kote_{fold_no_out}.csv"),
            index=False,
        )

    pd.DataFrame(performance_rows).to_csv(
        os.path.join(path1, "Performance_Summary.csv"),
        index=False,
    )

    pooled_df = pd.concat(pooled_rows, ignore_index=True)
    pooled_df.to_csv(
        os.path.join(path1, "All_Denormalized_Predictions_SVR.csv"),
        index=False,
    )

    pooled_metrics = [pooled_metrics_row(pooled_df, "All")]
    for fold_out, group in pooled_df.groupby("Fold_out"):
        pooled_metrics.append(
            pooled_metrics_row(group, "Fold_out", fold_out=fold_out)
        )

    pd.DataFrame(pooled_metrics).to_csv(
        os.path.join(path1, "Pooled_Denormalized_Metrics_SVR.csv"),
        index=False,
    )

In [ ]:
# Random Forest tuning, multiple targets, and pooled denormalized metrics (NoIntp+ManVa)

import os
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics as sm
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor


# Configuration
names = ["BMP", "COD", "NR", "PR"]

inpt_nums = {
    "BMP": 10,
    "COD": 10,
    "NR": 10,
    "PR": 10,
}

niter = 100      # 50 RF is slower than SVR; raise if time allows, initial 200
initp = 10      # initial 20

k_out = 5
fo_start = 1

parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv2/"


def make_regressor(n_estimators, max_depth, min_samples_split,
                   min_samples_leaf, max_features):
    return RandomForestRegressor(
        n_estimators=int(n_estimators),
        max_depth=int(max_depth),
        min_samples_split=int(min_samples_split),
        min_samples_leaf=int(min_samples_leaf),
        max_features=float(max_features),
        random_state=42,
        n_jobs=-1,
    )


def relative_error_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    error = actual - np.asarray(predicted, dtype=float)

    return np.divide(
        error * 100.0,
        actual,
        out=np.full_like(error, np.nan),
        where=actual != 0,
    )


def metric_values(actual, predicted):
    mse = sm.mean_squared_error(actual, predicted)

    return {
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(mse, 3),
        "RMSE": round(float(np.sqrt(mse)), 3),
        "R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
    }


def pooled_metrics_row(group, scope, fold_out="All"):
    actual = group["Actual(real)"].to_numpy(dtype=float)
    predicted = group["Prediction(real)"].to_numpy(dtype=float)
    residual = actual - predicted
    relative_error = relative_error_percent(actual, predicted)

    return {
        "Name": group["Name"].iloc[0],
        "Scope": scope,
        "Fold_out": fold_out,
        "N_predictions": len(group),
        "Pooled_R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(sm.mean_squared_error(actual, predicted), 3),
        "RE_mean": round(float(np.mean(residual)), 3),
        "RE%_mean": round(float(np.nanmean(relative_error)), 3),
    }


for name in names:
    print(f"===== Running {name} =====")
    inpt_num = inpt_nums[name]

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path1 = os.path.join(
        parent_dir,
        f"RF {name} Best Models_{timestamp}",
    )
    os.makedirs(path1, exist_ok=True)

    performance_rows = []
    pooled_rows = []

    for fold_no_out in range(fo_start, k_out + 1):
        # Outer held-out test data.
        data2 = pd.read_csv(os.path.join(
            parent_dir,
            f"Testing_data_ori_{name}-fold{fold_no_out}.csv",
        ))

        # Remaining original folds form the outer-training data.
        data = pd.read_csv(parent_dir + 'Training_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
        # extra_frames = []
        # for fold_num in range(1, k_out + 1):
        #     if fold_num == fold_no_out:
        #         continue
        #     extra_frames.append(pd.read_csv(os.path.join(
        #         parent_dir,
        #         f"Testing_data_ori_{name}-fold{fold_num}.csv",
        #     )))
        # data = pd.concat(extra_frames, axis=0, ignore_index=True)

        # Random 20% of outer-training data as validation.
        data, data3 = train_test_split(
            data,
            test_size=0.20,
            random_state=42,
            shuffle=True,
        )

        # Fit only on training data; transform all three datasets.
        scaler = StandardScaler()
        scaler.fit(data)

        dataset1 = scaler.transform(data)
        dataset2 = scaler.transform(data2)
        dataset3 = scaler.transform(data3)

        X_training = dataset1[:, :inpt_num]
        Y_training = dataset1[:, inpt_num]

        X_testing = dataset2[:, :inpt_num]
        Y_testing = dataset2[:, inpt_num]

        X_valid = dataset3[:, :inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # Bayesian optimization of RF hyperparameters (validation R2).
        def rf_objective(n_estimators, max_depth, min_samples_split,
                         min_samples_leaf, max_features):
            model = make_regressor(
                n_estimators, max_depth, min_samples_split,
                min_samples_leaf, max_features,
            )
            model.fit(X_training, Y_training)
            prediction = model.predict(X_valid)
            return sm.r2_score(Y_valid, prediction)

        optimizer = BayesianOptimization(
            f=rf_objective,
            pbounds={
                "n_estimators": (100, 1000),
                "max_depth": (2, 30),
                "min_samples_split": (2, 20),
                "min_samples_leaf": (1, 10),
                "max_features": (0.3, 1.0),
            },
            random_state=1,
        )
        optimizer.maximize(
            n_iter=niter,
            init_points=initp,
        )
        best_params = optimizer.max["params"]

        model = make_regressor(**best_params)
        model.fit(X_training, Y_training)
        prediction = model.predict(X_testing)

        model_name = f"RF_model_{name}_kote_{fold_no_out}"
        joblib.dump(
            model,
            os.path.join(path1, model_name + ".joblib"),
        )

        # Convert target values and predictions back to original units.
        ymean = scaler.mean_[inpt_num]
        yscale = scaler.scale_[inpt_num]

        actual_real = Y_testing * yscale + ymean
        predicted_real = prediction * yscale + ymean
        residual_real = actual_real - predicted_real
        residual_percent_real = relative_error_percent(
            actual_real,
            predicted_real,
        )

        normalized_metrics = metric_values(Y_testing, prediction)
        real_metrics = metric_values(actual_real, predicted_real)

        performance_rows.append({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "n_estimators": int(best_params["n_estimators"]),
            "max_depth": int(best_params["max_depth"]),
            "min_samples_split": int(best_params["min_samples_split"]),
            "min_samples_leaf": int(best_params["min_samples_leaf"]),
            "max_features": round(float(best_params["max_features"]), 3),
            "Val_R2": round(optimizer.max["target"], 3),
            **normalized_metrics,
            "MAE_real": real_metrics["MAE"],
            "MSE_real": real_metrics["MSE"],
            "RMSE_real": real_metrics["RMSE"],
            "R2_real": real_metrics["R2"],
            "RE_mean_real": round(float(np.mean(residual_real)), 3),
            "RE%_mean_real": round(float(np.nanmean(residual_percent_real)), 3),
        })

        predictions = pd.DataFrame({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "Test_row": np.arange(len(Y_testing)),
            "Actual": Y_testing,
            "Prediction": prediction,
            "Actual(real)": actual_real,
            "Prediction(real)": predicted_real,
            "RErr(real)": residual_real,
            "RErr%(real)": residual_percent_real,
        })
        pooled_rows.append(predictions)

        predictions.to_csv(
            os.path.join(path1, f"Predictions_{name}_kote_{fold_no_out}.csv"),
            index=False,
        )

    pd.DataFrame(performance_rows).to_csv(
        os.path.join(path1, "Performance_Summary.csv"),
        index=False,
    )

    pooled_df = pd.concat(pooled_rows, ignore_index=True)
    pooled_df.to_csv(
        os.path.join(path1, "All_Denormalized_Predictions_RF.csv"),
        index=False,
    )

    pooled_metrics = [pooled_metrics_row(pooled_df, "All")]
    for fold_out, group in pooled_df.groupby("Fold_out"):
        pooled_metrics.append(
            pooled_metrics_row(group, "Fold_out", fold_out=fold_out)
        )

    pd.DataFrame(pooled_metrics).to_csv(
        os.path.join(path1, "Pooled_Denormalized_Metrics_RF.csv"),
        index=False,
    )

In [ ]:
# Ridge tuning, multiple targets, and pooled denormalized metrics (NoIntp+ManVa)

import os
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics as sm
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge


# Configuration
names = ["BMP", "COD", "NR", "PR"]

inpt_nums = {
    "BMP": 10,
    "COD": 10,
    "NR": 10,
    "PR": 10,
}

niter = 200      # Ridge is very fast; raise freely
initp = 20

k_out = 5
fo_start = 1

parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv2/"


def make_regressor(log_alpha):
    # alpha is searched on a log10 scale
    return Ridge(alpha=10 ** log_alpha, random_state=42)


def relative_error_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    error = actual - np.asarray(predicted, dtype=float)

    return np.divide(
        error * 100.0,
        actual,
        out=np.full_like(error, np.nan),
        where=actual != 0,
    )


def metric_values(actual, predicted):
    mse = sm.mean_squared_error(actual, predicted)

    return {
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(mse, 3),
        "RMSE": round(float(np.sqrt(mse)), 3),
        "R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
    }


def pooled_metrics_row(group, scope, fold_out="All"):
    actual = group["Actual(real)"].to_numpy(dtype=float)
    predicted = group["Prediction(real)"].to_numpy(dtype=float)
    residual = actual - predicted
    relative_error = relative_error_percent(actual, predicted)

    return {
        "Name": group["Name"].iloc[0],
        "Scope": scope,
        "Fold_out": fold_out,
        "N_predictions": len(group),
        "Pooled_R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(sm.mean_squared_error(actual, predicted), 3),
        "RE_mean": round(float(np.mean(residual)), 3),
        "RE%_mean": round(float(np.nanmean(relative_error)), 3),
    }


for name in names:
    print(f"===== Running {name} =====")
    inpt_num = inpt_nums[name]

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path1 = os.path.join(
        parent_dir,
        f"Ridge {name} Best Models_{timestamp}",
    )
    os.makedirs(path1, exist_ok=True)

    performance_rows = []
    pooled_rows = []

    for fold_no_out in range(fo_start, k_out + 1):
        # Outer held-out test data.
        data2 = pd.read_csv(os.path.join(
            parent_dir,
            f"Testing_data_ori_{name}-fold{fold_no_out}.csv",
        ))

        # Remaining original folds form the outer-training data.
        data = pd.read_csv(parent_dir + 'Training_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
        # extra_frames = []
        # for fold_num in range(1, k_out + 1):
        #     if fold_num == fold_no_out:
        #         continue
        #     extra_frames.append(pd.read_csv(os.path.join(
        #         parent_dir,
        #         f"Testing_data_ori_{name}-fold{fold_num}.csv",
        #     )))
        # data = pd.concat(extra_frames, axis=0, ignore_index=True)

        # Random 20% of outer-training data as validation.
        data, data3 = train_test_split(
            data,
            test_size=0.20,
            random_state=42,
            shuffle=True,
        )

        # Fit only on training data; transform all three datasets.
        scaler = StandardScaler()
        scaler.fit(data)

        dataset1 = scaler.transform(data)
        dataset2 = scaler.transform(data2)
        dataset3 = scaler.transform(data3)

        X_training = dataset1[:, :inpt_num]
        Y_training = dataset1[:, inpt_num]

        X_testing = dataset2[:, :inpt_num]
        Y_testing = dataset2[:, inpt_num]

        X_valid = dataset3[:, :inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # Bayesian optimization of alpha (validation R2).
        def ridge_objective(log_alpha):
            model = make_regressor(log_alpha)
            model.fit(X_training, Y_training)
            prediction = model.predict(X_valid)
            return sm.r2_score(Y_valid, prediction)

        optimizer = BayesianOptimization(
            f=ridge_objective,
            pbounds={
                "log_alpha": (-4, 4),     # alpha: 1e-4 - 1e4
            },
            random_state=1,
        )
        optimizer.maximize(
            n_iter=niter,
            init_points=initp,
        )
        best_params = optimizer.max["params"]

        model = make_regressor(**best_params)
        model.fit(X_training, Y_training)
        prediction = model.predict(X_testing)

        model_name = f"Ridge_model_{name}_kote_{fold_no_out}"
        joblib.dump(
            model,
            os.path.join(path1, model_name + ".joblib"),
        )

        # Convert target values and predictions back to original units.
        ymean = scaler.mean_[inpt_num]
        yscale = scaler.scale_[inpt_num]

        actual_real = Y_testing * yscale + ymean
        predicted_real = prediction * yscale + ymean
        residual_real = actual_real - predicted_real
        residual_percent_real = relative_error_percent(
            actual_real,
            predicted_real,
        )

        normalized_metrics = metric_values(Y_testing, prediction)
        real_metrics = metric_values(actual_real, predicted_real)

        performance_rows.append({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "alpha": round(10 ** best_params["log_alpha"], 5),
            "Val_R2": round(optimizer.max["target"], 3),
            **normalized_metrics,
            "MAE_real": real_metrics["MAE"],
            "MSE_real": real_metrics["MSE"],
            "RMSE_real": real_metrics["RMSE"],
            "R2_real": real_metrics["R2"],
            "RE_mean_real": round(float(np.mean(residual_real)), 3),
            "RE%_mean_real": round(float(np.nanmean(residual_percent_real)), 3),
        })

        predictions = pd.DataFrame({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "Test_row": np.arange(len(Y_testing)),
            "Actual": Y_testing,
            "Prediction": prediction,
            "Actual(real)": actual_real,
            "Prediction(real)": predicted_real,
            "RErr(real)": residual_real,
            "RErr%(real)": residual_percent_real,
        })
        pooled_rows.append(predictions)

        predictions.to_csv(
            os.path.join(path1, f"Predictions_{name}_kote_{fold_no_out}.csv"),
            index=False,
        )

    pd.DataFrame(performance_rows).to_csv(
        os.path.join(path1, "Performance_Summary.csv"),
        index=False,
    )

    pooled_df = pd.concat(pooled_rows, ignore_index=True)
    pooled_df.to_csv(
        os.path.join(path1, "All_Denormalized_Predictions.csv"),
        index=False,
    )

    pooled_metrics = [pooled_metrics_row(pooled_df, "All")]
    for fold_out, group in pooled_df.groupby("Fold_out"):
        pooled_metrics.append(
            pooled_metrics_row(group, "Fold_out", fold_out=fold_out)
        )

    pd.DataFrame(pooled_metrics).to_csv(
        os.path.join(path1, "Pooled_Denormalized_Metrics.csv"),
        index=False,
    )

In [ ]:
# =====================================================================================================
# Permutation (y-scrambling) test for the 5-fold CV DNN, COD target  -  DWT-D-26-01475, round 2
# Paste into ONE Colab cell (or run as a script). Resumable: results are appended to a CSV in Drive,
# so after a disconnect just run the cell again and it continues where it stopped.
#
# Procedure (identical to the permutation tests of the other models):
#   * the same 5 folds as the reported run (your Training_/Testing_data_ori_COD-fold{k}.csv files,
#     already imputed per fold);
#   * in every fold the architecture selected by the tuner in that fold is re-used (no re-tuning);
#   * step 0 trains on the REAL target ("replica R2"); steps 1..N_PERM train on a target that is shuffled
#     across all 45 records (one shuffle per permutation, applied consistently to every fold);
#   * pooled out-of-fold R2 for each run;  p = (1 + #(null R2 >= replica R2)) / (1 + N_PERM).
#
# >>> The only thing you have to edit is  train_and_predict()  (section 2): put in your own Keras
#     preprocessing / model / callbacks / epochs / batch size exactly as in your 5-fold runs.
# =====================================================================================================
import os, time
import numpy as np, pandas as pd

# ----------------------------------------------------------------------------- 1. configuration
BASE ="E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv2/"
# BASE = '/content/drive/MyDrive/Colab Notebooks/AMBR Prediction_v5kcv2'   # folder with the fold files
OUT = os.path.join(BASE, 'Permutation_COD_DNN_5fold.csv')                # results (one row per run)
N_PERM = 49 #49, 99
SEED = 0

# # try:                                   # mount Drive when running in Colab
#     from google.colab import drive
#     drive.mount('/content/drive')
# except ImportError:
#     pass

# fold of each COD record (original record order, 45 records) - from fold_assignment_COD.csv
FOLD = np.array([4, 5, 4, 5, 4, 3, 3, 4, 1, 5, 2, 1, 4, 3, 2, 1, 2, 1, 3, 5, 3, 5, 1, 5, 5, 4, 3, 1, 1, 3,
                 1, 2, 2, 2, 4, 2, 1, 2, 3, 5, 4, 2, 4, 3, 5])
# architecture selected by the tuner in each fold of the reported run (from the saved .keras files)
ARCH = {1: [80, 32, 40, 64], 2: [24, 72], 3: [24, 24, 8, 96], 4: [24, 40, 24, 16], 5: [24, 72, 112, 96]}

# Training file of fold k = records with FOLD != k in original order; testing file = records with FOLD == k.
# (checked: the target columns of your files line up exactly with this mapping)
TR = {k: pd.read_csv(os.path.join(BASE, f'Training_data_ori_COD-fold{k}.csv'), encoding='latin-1') for k in range(1, 6)}
TE = {k: pd.read_csv(os.path.join(BASE, f'Testing_data_ori_COD-fold{k}.csv'), encoding='latin-1') for k in range(1, 6)}
IDX_TR = {k: np.where(FOLD != k)[0] for k in range(1, 6)}
IDX_TE = {k: np.where(FOLD == k)[0] for k in range(1, 6)}
Y = np.empty(len(FOLD))
for k in range(1, 6):
    Y[IDX_TE[k]] = TE[k].iloc[:, -1].to_numpy(float)
for k in range(1, 6):                    # sanity check of the mapping
    assert np.allclose(Y[IDX_TR[k]], TR[k].iloc[:, -1].to_numpy(float)), f'fold {k}: row order mismatch'
print('COD records:', len(Y), '| folds:', {k: (len(IDX_TR[k]), len(IDX_TE[k])) for k in IDX_TR})


# ----------------------------------------------------------------------------- 2. YOUR MODEL  (edit here)
def train_and_predict(X_train, y_train, X_test, units, seed):
    """Train one DNN on (X_train, y_train) and return predictions for X_test in REAL units (mg/L).
    Replace the body with your own code from the 5-fold runs: same scaling, 80/20 fit/validation split,
    layers, optimizer (Adam, lr = 0.01), loss, callbacks (EarlyStopping / ReduceLROnPlateau ...),
    epochs and batch size. Everything must be fitted on X_train / y_train only."""
    import tensorflow as tf
    from tensorflow import keras
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split

    keras.utils.set_random_seed(seed)
    sx, sy = StandardScaler().fit(X_train), StandardScaler().fit(y_train.reshape(-1, 1))
    Xs, ys = sx.transform(X_train), sy.transform(y_train.reshape(-1, 1)).ravel()
    X_fit, X_val, y_fit, y_val = train_test_split(Xs, ys, test_size=0.2, random_state=seed)   # 80/20

    model = keras.Sequential([keras.layers.Input(shape=(X_train.shape[1],))] +
                             [keras.layers.Dense(u, activation='relu') for u in units] +
                             [keras.layers.Dense(1)])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-2), loss='mse')
    callbacks = [  # <<< REPLACE with your callbacks (patience, factor, min_lr ...)
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=20, min_lr=1e-4),
    ]
    model.fit(X_fit, y_fit, validation_data=(X_val, y_val), epochs=1000, batch_size=8,   # <<< your values
              callbacks=callbacks, verbose=0)
    p = model.predict(sx.transform(X_test), verbose=0).ravel()
    keras.backend.clear_session()
    return sy.inverse_transform(p.reshape(-1, 1)).ravel()


# ----------------------------------------------------------------------------- 3. one 5-fold run
def r2(y, p):
    return 1 - np.sum((y - p) ** 2) / np.sum((y - np.mean(y)) ** 2)


def run_5fold(y_all, seed):
    """Pooled out-of-fold R2 of the whole 5-fold procedure for target vector y_all (original record order)."""
    pred = np.empty_like(y_all)
    for k in range(1, 6):
        Xtr = TR[k].iloc[:, :-1].to_numpy(float)
        Xte = TE[k].iloc[:, :-1].to_numpy(float)
        pred[IDX_TE[k]] = train_and_predict(Xtr, y_all[IDX_TR[k]], Xte, ARCH[k], seed + k)
    return r2(y_all, pred)


# ----------------------------------------------------------------------------- 4. replica + permutations
done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame(columns=['run', 'R2'])
rng = np.random.default_rng(SEED)
perms = [rng.permutation(len(Y)) for _ in range(N_PERM)]       # fixed in advance -> resumable

for run in range(N_PERM + 1):                                   # run 0 = real target
    if run in set(done.run.astype(int)):
        continue
    t0 = time.time()
    y_run = Y if run == 0 else Y[perms[run - 1]]
    val = run_5fold(y_run, SEED + 100 * run)
    done = pd.concat([done, pd.DataFrame([dict(run=run, R2=val)])], ignore_index=True)
    done.to_csv(OUT, index=False)
    print(f'run {run:3d} {"(real) " if run == 0 else "(perm) "} R2 = {val:7.3f}   ({time.time() - t0:.0f} s)')

# ----------------------------------------------------------------------------- 5. result
real = float(done.loc[done.run == 0, 'R2'].iloc[0])
null = done.loc[done.run > 0, 'R2'].to_numpy(float)
p = (1 + np.sum(null >= real)) / (1 + len(null))
print(f'\nreported R2 0.241 | replica R2 {real:.3f} | n_perm {len(null)} | null mean {null.mean():.3f} '
      f'| null 95th pct {np.percentile(null, 95):.3f} | null max {null.max():.3f} | p = {p:.3f}')
